# Notebook A v4.2: Research Actions + Signal Weights

**Run order:** First. Before Notebook B v4.2 and Notebook D v4.3.

## v4.2 changes vs v4.1

DNA signal support added:

- Three new applicability flag columns added to `ref_signal_weights`:
  - `requires_dna_match_tag` — signal only applicable if person has `DNA Match` tag (`@T10@`)
  - `requires_dna_ancestor_tag` — signal only applicable if person has `Common DNA Ancestor` tag (`@T5@`)
  - `requires_generation_lte` — signal only applicable if generation depth <= N
- New action: `INVESTIGATE_DNA_MATCH`
- Four new signals:
  - `SIGNAL_DNA_CITATION_MISSING` (completeness, 20) — DNA Match tag present, no Ancestry citation in tree
  - `SIGNAL_DNA_PATH_INCOMPLETE` (completeness, 25) — DNA Match tag present, path to researcher has missing Connection/Ancestor tags
  - `SIGNAL_NO_DNA_CORROBORATION` (completeness, 25) — direct ancestor gen 1-6, no Common DNA Ancestor tag
  - `SIGNAL_COMMON_ANCESTOR_UNVERIFIED` (evidence, 20) — Common DNA Ancestor tag, no DNA Match person's path runs through them

## v4.1 changes vs v4

Action code cleanup — retired two vague, non-closeable actions and replaced with
specific alternatives:

- `CLUSTER_SOURCES` retired — signals that previously mapped to this now have no
  primary action (`SIGNAL_VERY_LOW_EVIDENCE_DENSITY`, `SIGNAL_LOW_EVIDENCE_DENSITY`)
  or a more specific one. Signals still fire and score; they just do not surface
  as a This Week action.
- `REVIEW_EXISTING_SOURCES` retired — replaced by two new specific action codes:
  - `RECONCILE_TRANSCRIPT_FACTS` — update tree with facts extracted from transcripts
  - `ORDER_BMD_CERTIFICATE` — order a GRO/ScotlandsPeople certificate
- Several signals now intentionally have no primary action mapping
  (`SIGNAL_VERY_LOW_EVIDENCE_DENSITY`, `SIGNAL_LOW_EVIDENCE_DENSITY`,
  `SIGNAL_SINGLE_SOURCE_DEPENDENCE`, `SIGNAL_NEWSPAPER_MENTION`,
  `SIGNAL_WILL_OR_PROBATE`, `SIGNAL_TRANSCRIPT_RICH`) — these contribute
  to scoring but are too generic or narrative to drive a closeable task.

## v4 changes vs v3

`ref_signal_weights` gains applicability flag columns. These replace the inline
`CASE WHEN` applicability logic that was hardcoded in Notebook D v3, which required
editing the scoring view every time a signal was added.

With v4, adding a new signal requires:
1. Add detection boolean to Notebook B (signals view)
2. Insert one row into `ref_signal_weights` with applicability flags set

`gold_person_scores` and `gold_person_signal_scores` pick it up automatically.

## Applicability flag columns

| Column | Type | Meaning |
|---|---|---|
| `applies_always` | BOOLEAN | In denominator for all people (overrides other flags if TRUE) |
| `dimension` | STRING | completeness / evidence / narrative |
| `min_birth_year` | INT | Only applicable if birth_year >= N |
| `max_birth_year` | INT | Only applicable if birth_year <= N (NULL = no upper limit) |
| `sex_restriction` | STRING | 'M' or 'F'; NULL = any sex |
| `requires_expected_to_die` | BOOLEAN | birth_year <= 1930 AND expected_end_year < current_year |
| `requires_lifespan_gte` | INT | effective_span_years >= N |
| `requires_survived_past_40` | BOOLEAN | expected_end_year >= birth_year + 40 |
| `requires_working_age` | BOOLEAN | expected_end_year >= birth_year + 18 (male occupation check) |
| `requires_census_years` | BOOLEAN | At least one expected census year > 0 |
| `requires_post_1837_event` | BOOLEAN | birth_year >= 1837 OR death_year >= 1837 OR marriage >= 1837 |
| `requires_has_document` | BOOLEAN | Has >= 1 matched file in silver_document_person |
| `requires_transcript` | BOOLEAN | Has >= 1 transcribed non-BMDIndex document |
| `requires_citations` | BOOLEAN | Has >= 1 citation in gold_source_coverage |
| `requires_family_events` | BOOLEAN | Has marriages or child births |
| `requires_proximity_lte` | INT | proximity <= N |
| `requires_total_facts_gte` | INT | total_facts >= N |
| `requires_death_recorded` | BOOLEAN | death_year IS NOT NULL |
| `not_young_death` | BOOLEAN | Suppress if effective_span_years BETWEEN 16 AND 40 |
| `requires_dna_match_tag` | BOOLEAN | Person has DNA Match tag (@T10@) — NEW in v4.2 |
| `requires_dna_ancestor_tag` | BOOLEAN | Person has Common DNA Ancestor tag (@T5@) — NEW in v4.2 |
| `requires_generation_lte` | INT | Generation depth <= N — NEW in v4.2 |


In [0]:

%sql
-- ============================================================
-- v4.2 changes:
--   ADDED: INVESTIGATE_DNA_MATCH
--     Research a DNA match — add/verify tags and citation in tree.
--     Primary action for all four DNA signals.
-- ============================================================

CREATE OR REPLACE TABLE genealogy.gold_research_action (
  action_code     STRING,
  action_label    STRING,
  action_category STRING,
  default_effort  INT
);

INSERT INTO genealogy.gold_research_action VALUES
('RESOLVE_NAME_VARIANTS',        'Resolve name variants / aliases',                     'identity',  2),
('RESOLVE_CONFLICTS',            'Investigate and resolve transcript vs tree conflicts', 'evidence',  2),
('RECONCILE_TRANSCRIPT_FACTS',   'Update tree from transcript facts',                   'evidence',  2),
('ORDER_BMD_CERTIFICATE',        'Order BMD certificate',                               'evidence',  2),
('SEARCH_BIRTH_RECORDS',         'Search for birth/baptism records',                    'lifecycle', 1),
('SEARCH_DEATH_RECORDS',         'Search for death/burial records',                     'lifecycle', 1),
('SEARCH_BURIAL_RECORDS',        'Search for burial/cremation records',                 'lifecycle', 1),
('SEARCH_MARRIAGE_RECORDS',      'Search for marriage records',                         'lifecycle', 1),
('SEARCH_CENSUS',                'Search census / population records',                  'lifecycle', 1),
('DOWNLOAD_DOCUMENTS',           'Download cited documents from Ancestry/archive',      'evidence',  2),
('TRANSCRIBE_DOCUMENTS',         'Run OCR transcription pipeline on downloaded docs',   'evidence',  1),
('TRACE_PARENTS',                'Identify or verify parents',                          'family',    2),
('TRACE_CHILDREN_FORWARD',       'Trace children forward',                              'family',    2),
('VERIFY_FAMILY_EVENTS',         'Source family events (marriage, children)',            'family',    1),
('INVESTIGATE_MIGRATION',        'Investigate migration and movement',                  'narrative', 2),
('SEARCH_MILITARY_RECORDS',      'Search military service records',                     'narrative', 2),
('INVESTIGATE_MILITARY',         'Investigate and record confirmed military service',   'narrative', 3),
('INVESTIGATE_OCCUPATION',       'Research occupational history and records',           'narrative', 2),
('INVESTIGATE_LATER_LIFE',       'Search for late-life events and records',             'narrative', 2),
('INVESTIGATE_DNA_MATCH',        'Research and link DNA match in tree',                 'dna',       2);


In [0]:

%sql
CREATE OR REPLACE TABLE genealogy.ref_intent_category_weights (
  intent   STRING,
  category STRING,
  weight   DOUBLE
);

INSERT OVERWRITE genealogy.ref_intent_category_weights VALUES
  ('integrity', 'evidence',     0.60),
  ('integrity', 'completeness', 0.40),
  ('narrative', 'texture',      0.45),
  ('narrative', 'family',       0.30),
  ('narrative', 'context',      0.25);


In [0]:

%sql
-- ============================================================
-- CELL 3: ref_signal_weights (v4.2)
--
-- v4.2 changes vs v4.1:
--   THREE NEW flag columns added (columns 19-21):
--     requires_dna_match_tag     — only applicable for DNA Match tagged people
--     requires_dna_ancestor_tag  — only applicable for Common DNA Ancestor tagged people
--     requires_generation_lte    — only applicable if generation depth <= N
--
--   FOUR NEW signal rows added at end (DNA signals):
--     SIGNAL_DNA_CITATION_MISSING
--     SIGNAL_DNA_PATH_INCOMPLETE
--     SIGNAL_NO_DNA_CORROBORATION
--     SIGNAL_COMMON_ANCESTOR_UNVERIFIED
--
-- NULL in a flag column means "restriction does not apply".
-- Non-NULL value means "this restriction must be satisfied".
--
-- applies_always = TRUE means: ignore all other flags, always in denominator.
-- applies_always = FALSE means: evaluate the other flags.
--
-- LOW and VERY_LOW evidence density are mutually exclusive bands:
--   VERY_LOW: avg_spf < 0.3 or NULL  → applies_always TRUE
--   LOW:      avg_spf in [0.3, 1.0)  → no restriction (always in denom)
-- Both are always in the denominator. Only one can fire at a time.
-- ============================================================

CREATE OR REPLACE TABLE genealogy.ref_signal_weights (
  signal_code               STRING,
  category                  STRING,
  intent                    STRING,
  base_score                INT,
  reason_label              STRING,
  rationale                 STRING,
  -- dimension: completeness | evidence | narrative
  dimension                 STRING,
  -- applicability flags
  applies_always            BOOLEAN, --01
  min_birth_year            INT,     --02
  max_birth_year            INT,     --03
  sex_restriction           STRING,  --04
  requires_expected_to_die  BOOLEAN, --05
  requires_lifespan_gte     INT,     --06
  requires_survived_past_40 BOOLEAN, --07
  requires_working_age      BOOLEAN, --08
  requires_census_years     BOOLEAN, --09
  requires_post_1837_event  BOOLEAN, --10
  requires_has_document     BOOLEAN, --11
  requires_transcript       BOOLEAN, --12
  requires_citations        BOOLEAN, --13
  requires_family_events    BOOLEAN, --14
  requires_proximity_lte    INT,     --15
  requires_total_facts_gte  INT,     --16
  requires_death_recorded   BOOLEAN, --17
  not_young_death           BOOLEAN, --18
  -- NEW in v4.2: DNA applicability flags
  requires_dna_match_tag    BOOLEAN, --19
  requires_dna_ancestor_tag BOOLEAN, --20
  requires_generation_lte   INT      --21
);

INSERT OVERWRITE genealogy.ref_signal_weights VALUES

-- ================================================================
-- COMPLETENESS signals
-- ================================================================

-- NO_BIRTH_RECORDED: always applicable
('SIGNAL_NO_BIRTH_RECORDED', 'completeness', 'integrity', 30,
 'birth not recorded', NULL, 'completeness',
 TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),


-- NO_DEATH_RECORDED: birth_year <= 1930 and expected to have died by now
('SIGNAL_NO_DEATH_RECORDED', 'completeness', 'integrity', 20,
 'death not recorded', NULL, 'completeness',
 FALSE, NULL, 1930, NULL, TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),


-- NO_MARRIAGES: lifespan >= 16 (could have married); suppress for young deaths
('SIGNAL_NO_MARRIAGES', 'completeness', 'integrity', 15,
 'no marriage recorded', 'age-guarded: suppressed for young deaths', 'completeness',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL, NULL),


-- NO_CHILDREN: proximity <= 1 only; suppress for young deaths
('SIGNAL_NO_CHILDREN', 'completeness', 'integrity', 10,
 'no children recorded', 'proximity guard: proximity <= 1 only', 'completeness',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 1, NULL, NULL, TRUE, NULL, NULL, NULL),


-- MISSING_PARENT: always applicable
('SIGNAL_MISSING_PARENT', 'completeness', 'integrity', 25,
 'missing parent', NULL, 'completeness',
 TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),


-- MISSING_CENSUS_COVERAGE: at least one expected census year
('SIGNAL_MISSING_CENSUS_COVERAGE', 'completeness', 'integrity', 20,
 'missing expected census', 'person was alive during census year but no RESI event present', 'completeness',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),


-- UNCOVERED_SOURCES: has at least one citation
('SIGNAL_UNCOVERED_SOURCES', 'completeness', 'integrity', 20,
 'uncovered cited sources', 'cited sources with no downloaded document', 'completeness',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),


-- DOCS_NOT_TRANSCRIBED: has at least one matched document
('SIGNAL_DOCS_NOT_TRANSCRIBED', 'completeness', 'integrity', 15,
 'documents not transcribed', 'evidence downloaded but not yet verified via OCR', 'completeness',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),


-- LATE_LIFE_GAP: expected to have survived past 40
('SIGNAL_LATE_LIFE_GAP', 'completeness', 'integrity', 15,
 'late-life records gap', 'no events recorded after age 40 despite expected survival', 'completeness',
 FALSE, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),


-- EARLY_LIFE_ONLY: has family events (so adult records should exist)
('SIGNAL_EARLY_LIFE_ONLY', 'completeness', 'integrity', 10,
 'early-life records only', 'classic birth+parents-only profile, no adult records', 'completeness',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL),


-- CHILD_GAPS: always applicable (fires only when relevant)
('SIGNAL_CHILD_GAPS', 'completeness', 'integrity', 10,
 'gaps between child births', 'unusually long inter-birth gap or marriage with no children', 'completeness',
 TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),


-- UNCONFIRMED_MILITARY: born 1867-1928
('SIGNAL_UNCONFIRMED_MILITARY', 'completeness', 'integrity', 15,
 'military service unconfirmed', 'born 1867-1928 — WWI/WWII service neither confirmed nor denied', 'completeness',
 FALSE, 1867, 1929, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),


-- MISSING_OCCUPATION: male, lived to working age
('SIGNAL_MISSING_OCCUPATION', 'completeness', 'integrity', 10,
 'no occupation recorded', 'male with working-age lifespan and zero occupation events recorded', 'completeness',
 FALSE, NULL, NULL, 'M', NULL, NULL, NULL, TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),


-- MISSING_BURIAL: always applies
('SIGNAL_MISSING_BURIAL', 'completeness', 'integrity', 8,
 'burial not recorded', 'post-1837 death with no burial or cremation event', 'completeness',
 TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),


-- NO_RESIDENCE: lifespan >= 20 years
('SIGNAL_NO_RESIDENCE', 'completeness', 'integrity', 10,
 'no residence events', 'lifespan >= 20 years with zero residence/census events', 'completeness',
 FALSE, NULL, NULL, NULL, NULL, 20, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),


-- ================================================================
-- EVIDENCE signals
-- ================================================================

-- NO_DOCUMENTS_AT_ALL: birth_year >= 1600 (pre-1600 may have no surviving docs)
('SIGNAL_NO_DOCUMENTS_AT_ALL', 'evidence', 'integrity', 40,
 'no documents filed', 'zero matched files — profile entirely uncorroborated by downloaded evidence', 'evidence',
 FALSE, 1600, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),


-- FACT_CONFLICT: has at least one transcript
('SIGNAL_FACT_CONFLICT', 'evidence', 'integrity', 30,
 'transcript conflicts tree', 'document contradicts recorded fact', 'evidence',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),


-- TRANSCRIPT_ONLY_FACTS: has at least one transcript
('SIGNAL_TRANSCRIPT_ONLY_FACTS', 'evidence', 'integrity', 20,
 'transcript facts not in tree', 'document contains facts not yet reflected in tree', 'evidence',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),


-- VERY_LOW_EVIDENCE_DENSITY: always applicable (catches NULL avg_spf)
('SIGNAL_VERY_LOW_EVIDENCE_DENSITY', 'evidence', 'integrity', 30,
 'very low source density (mostly unsourced)', 'avg sources per fact < 0.3 — profile largely unsourced', 'evidence',
 TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),


-- LOW_EVIDENCE_DENSITY: always applicable (mutually exclusive with VERY_LOW at fire time)
('SIGNAL_LOW_EVIDENCE_DENSITY', 'evidence', 'integrity', 15,
 'low source density (some unsourced facts)', 'avg sources per fact < 1.0', 'evidence',
 TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),


-- SINGLE_SOURCE_DEPENDENCE: total_facts >= 3
('SIGNAL_SINGLE_SOURCE_DEPENDENCE', 'evidence', 'integrity', 25,
 'single-source reliance', 'catches illusion-of-certainty problem', 'evidence',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 3, NULL, NULL, NULL, NULL, NULL),

-- UNSOURCED_FAMILY_EVENTS: always applicable
('SIGNAL_UNSOURCED_FAMILY_EVENTS', 'evidence', 'integrity', 25,
 'unsourced family events', 'marriages and child births without sources', 'evidence',
 TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),


-- IMPRECISE_DATES: post-1837 birth/death/marriage
('SIGNAL_IMPRECISE_DATES', 'evidence', 'integrity', 10,
 'imprecise dates', 'post-1837 civil registration events where a certificate should exist', 'evidence',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),


-- INCOMPLETE_NAME: always applicable
('SIGNAL_INCOMPLETE_NAME', 'evidence', 'integrity', 10,
 'incomplete name', 'suggests exhaustive searches not done', 'evidence',
 TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),


-- IMPRECISE_PLACES: birth_year >= 1600
('SIGNAL_IMPRECISE_PLACES', 'evidence', 'integrity', 10,
 'imprecise places', 'post-1837 records should resolve to town/parish level', 'evidence',
 FALSE, 1600, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),


-- ================================================================
-- NARRATIVE signals
-- ================================================================

('SIGNAL_MIGRANT',            'texture', 'narrative', 30, 'evidence of migration',          NULL, 'narrative', TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),

('SIGNAL_CONFIRMED_MILITARY', 'texture', 'narrative', 25, 'confirmed military service',      'has _MILT event or MilitaryRecord doc', 'narrative', TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),

('SIGNAL_YOUNG_DEATH',        'texture', 'narrative', 20, 'young death',                     NULL, 'narrative', TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),

('SIGNAL_NEWSPAPER_MENTION',  'texture', 'narrative', 25, 'newspaper mention',               'rare: newspaper clipping filed', 'narrative', TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),

('SIGNAL_WILL_OR_PROBATE',    'texture', 'narrative', 20, 'will or probate record',          'wealth/family dynamics', 'narrative', TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),

('SIGNAL_LARGE_FAMILY',       'texture', 'narrative', 15, 'large family (>= 6 children)',    'social history angle', 'narrative', TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),

('SIGNAL_STORY_WRITTEN',      'texture', 'narrative', -50,'story already written',           'suppresses narrative priority', 'narrative', TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),

('SIGNAL_MULTIPLE_SPOUSES',   'family',  'narrative', 40, 'multiple spouses',                NULL, 'narrative', TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),

('SIGNAL_TRANSCRIPT_RICH',    'context', 'narrative', 25, 'rich primary source material',    '>= 2 non-BMDIndex transcribed docs', 'narrative', TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),

('SIGNAL_VARIED_OCCUPATIONS', 'texture', 'narrative', 20, 'varied occupational history',     NULL, 'narrative', TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),

-- ================================================================
-- RESEARCH HINT signals (do not contribute to scoring)
-- ================================================================

('SIGNAL_POSSIBLE_MARRIAGE',  'completeness', 'integrity', 0, 'possible unrecorded marriage',    'female in 1939 register with no marriage recorded', 'narrative', FALSE, NULL, NULL, 'F', NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),

('SIGNAL_POSSIBLE_CHILDREN',  'completeness', 'integrity', 0, 'possible unrecorded children',    'female in 1911 census born <=1895', 'narrative', FALSE, NULL, 1895, 'F', NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL),

('SIGNAL_POSSIBLE_RESIDENCE', 'completeness', 'integrity', 0, 'residence records likely findable', NULL, 'narrative', FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL, NULL, NULL, NULL, NULL, NULL),

-- ================================================================
-- DNA signals (NEW in v4.2)
-- ================================================================
-- All DNA signals map to INVESTIGATE_DNA_MATCH (action_weight 3).
-- Applicability is tag-based — signals only enter the denominator
-- for people where they could meaningfully fire.
-- DNA match tagged people: requires_dna_match_tag = TRUE
-- Common DNA Ancestor tagged people: requires_dna_ancestor_tag = TRUE
-- Direct ancestors gen 1-6: requires_proximity_lte = 0, requires_generation_lte = 6
-- ================================================================

-- DNA_CITATION_MISSING: only applicable for DNA Match tagged people
-- Fires when a DNA Match tag exists but no Ancestry kit citation is in silver_person_source.
('SIGNAL_DNA_CITATION_MISSING', 'completeness', 'integrity', 20,
 'DNA match citation missing', 'DNA Match tag present but no Ancestry citation in tree', 'completeness',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL),

-- DNA_PATH_INCOMPLETE: only applicable for DNA Match tagged people
-- Fires when the path from the match to the researcher has people missing
-- DNA Connection tags, or the terminal ancestor is missing Common DNA Ancestor tag.
('SIGNAL_DNA_PATH_INCOMPLETE', 'completeness', 'integrity', 25,
 'DNA path tags incomplete', 'Path to researcher has missing DNA Connection or Common Ancestor tags', 'completeness',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL, NULL),

-- NO_DNA_CORROBORATION: only applicable for direct ancestors (proximity=0), gen depth 1-6
-- Fires when a direct ancestor in generations 1-6 has no Common DNA Ancestor tag.
-- Gen 0 excluded (researcher themselves). Gen > 6 not expected to have DNA evidence.
('SIGNAL_NO_DNA_CORROBORATION', 'completeness', 'integrity', 25,
 'ancestor not DNA-corroborated', 'Direct ancestor gen 1-6 with no Common DNA Ancestor tag', 'completeness',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, 0, NULL, NULL, NULL, NULL, NULL, 6),

-- COMMON_ANCESTOR_UNVERIFIED: only applicable for Common DNA Ancestor tagged people
-- Fires when a Common DNA Ancestor tag exists but no DNA Match person has a path
-- running through this person — the connection is claimed but not corroborated.
('SIGNAL_COMMON_ANCESTOR_UNVERIFIED', 'evidence', 'integrity', 20,
 'common ancestor unverified', 'Common DNA Ancestor tag present but no DNA match paths through this person', 'evidence',
 FALSE, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL, TRUE, NULL);


In [0]:

%sql
-- ============================================================
-- CELL 4: gold_research_signal_action (v4.2)
--
-- v4.2 changes vs v4.1:
--   ADDED mappings for 4 new DNA signals → INVESTIGATE_DNA_MATCH (weight 3)
-- ============================================================

CREATE OR REPLACE TABLE genealogy.gold_research_signal_action (
  signal_code   STRING,
  action_code   STRING,
  action_weight INT
);

INSERT INTO genealogy.gold_research_signal_action VALUES
  -- Evidence signals
  ('SIGNAL_NO_DOCUMENTS_AT_ALL',       'DOWNLOAD_DOCUMENTS',          3),
  ('SIGNAL_TRANSCRIPT_ONLY_FACTS',     'RECONCILE_TRANSCRIPT_FACTS',  3),
  ('SIGNAL_FACT_CONFLICT',             'RESOLVE_CONFLICTS',           3),
  ('SIGNAL_DOCS_NOT_TRANSCRIBED',      'TRANSCRIBE_DOCUMENTS',        3),
  ('SIGNAL_UNCOVERED_SOURCES',         'DOWNLOAD_DOCUMENTS',          3),
  ('SIGNAL_LOW_EVIDENCE_DENSITY',      'SEARCH_CENSUS',               2),
  ('SIGNAL_VERY_LOW_EVIDENCE_DENSITY', 'SEARCH_CENSUS',               2),
  -- SIGNAL_SINGLE_SOURCE_DEPENDENCE: no action — scores but too generic to drive a task
  ('SIGNAL_UNSOURCED_FAMILY_EVENTS',   'VERIFY_FAMILY_EVENTS',        3),
  ('SIGNAL_INCOMPLETE_NAME',           'SEARCH_MARRIAGE_RECORDS',     3),
  ('SIGNAL_INCOMPLETE_NAME',           'SEARCH_BIRTH_RECORDS',        2),
  ('SIGNAL_IMPRECISE_DATES',           'ORDER_BMD_CERTIFICATE',       3),
  ('SIGNAL_IMPRECISE_PLACES',          'ORDER_BMD_CERTIFICATE',       3),
  -- Completeness signals
  ('SIGNAL_NO_BIRTH_RECORDED',         'SEARCH_BIRTH_RECORDS',        3),
  ('SIGNAL_NO_DEATH_RECORDED',         'SEARCH_DEATH_RECORDS',        3),
  ('SIGNAL_NO_MARRIAGES',              'SEARCH_MARRIAGE_RECORDS',     3),
  ('SIGNAL_NO_CHILDREN',               'SEARCH_CENSUS',               3),
  ('SIGNAL_NO_CHILDREN',               'SEARCH_BIRTH_RECORDS',        2),
  ('SIGNAL_MISSING_CENSUS_COVERAGE',   'SEARCH_CENSUS',               3),
  ('SIGNAL_LATE_LIFE_GAP',             'INVESTIGATE_LATER_LIFE',      3),
  ('SIGNAL_LATE_LIFE_GAP',             'SEARCH_DEATH_RECORDS',        2),
  ('SIGNAL_EARLY_LIFE_ONLY',           'SEARCH_CENSUS',               3),
  ('SIGNAL_CHILD_GAPS',                'SEARCH_BIRTH_RECORDS',        3),
  ('SIGNAL_CHILD_GAPS',                'SEARCH_CENSUS',               2),
  ('SIGNAL_MISSING_BURIAL',            'SEARCH_BURIAL_RECORDS',       3),
  ('SIGNAL_MISSING_BURIAL',            'SEARCH_DEATH_RECORDS',        2),
  ('SIGNAL_NO_RESIDENCE',              'SEARCH_CENSUS',               3),
  ('SIGNAL_MISSING_PARENT',            'TRACE_PARENTS',               3),
  ('SIGNAL_MULTIPLE_SPOUSES',          'VERIFY_FAMILY_EVENTS',        3),
  ('SIGNAL_LARGE_FAMILY',              'TRACE_CHILDREN_FORWARD',      3),
  ('SIGNAL_LARGE_FAMILY',              'SEARCH_BIRTH_RECORDS',        2),
  ('SIGNAL_UNCONFIRMED_MILITARY',      'SEARCH_MILITARY_RECORDS',     3),
  ('SIGNAL_MISSING_OCCUPATION',        'SEARCH_CENSUS',               3),
  ('SIGNAL_MISSING_OCCUPATION',        'INVESTIGATE_OCCUPATION',      2),
  -- Narrative signals
  ('SIGNAL_CONFIRMED_MILITARY',        'INVESTIGATE_MILITARY',        3),
  ('SIGNAL_VARIED_OCCUPATIONS',        'INVESTIGATE_OCCUPATION',      3),
  ('SIGNAL_MIGRANT',                   'INVESTIGATE_MIGRATION',       3),
  ('SIGNAL_YOUNG_DEATH',               'SEARCH_DEATH_RECORDS',        3),
  -- SIGNAL_NEWSPAPER_MENTION: no action — narrative signal, no closeable task
  -- SIGNAL_WILL_OR_PROBATE: no action — same reasoning
  -- SIGNAL_TRANSCRIPT_RICH: no action — boosts story score only
  ('SIGNAL_POSSIBLE_MARRIAGE',         'SEARCH_MARRIAGE_RECORDS',     3),
  ('SIGNAL_POSSIBLE_MARRIAGE',         'SEARCH_CENSUS',               2),
  ('SIGNAL_POSSIBLE_CHILDREN',         'SEARCH_BIRTH_RECORDS',        3),
  ('SIGNAL_POSSIBLE_CHILDREN',         'SEARCH_CENSUS',               2),
  ('SIGNAL_POSSIBLE_RESIDENCE',        'SEARCH_CENSUS',               3),
  -- SIGNAL_STORY_WRITTEN: suppression signal — no actions
  -- DNA signals (NEW in v4.2)
  ('SIGNAL_DNA_CITATION_MISSING',      'INVESTIGATE_DNA_MATCH',       3),
  ('SIGNAL_DNA_PATH_INCOMPLETE',       'INVESTIGATE_DNA_MATCH',       3),
  ('SIGNAL_NO_DNA_CORROBORATION',      'INVESTIGATE_DNA_MATCH',       3),
  ('SIGNAL_COMMON_ANCESTOR_UNVERIFIED','INVESTIGATE_DNA_MATCH',       3);


In [0]:

%sql
-- 1. Signals with no action mapping
-- The following intentionally have no mapping — they score but don't drive a closeable task:
--   SIGNAL_STORY_WRITTEN: suppression only
--   SIGNAL_TRANSCRIPT_RICH, SIGNAL_NEWSPAPER_MENTION, SIGNAL_WILL_OR_PROBATE: narrative boosters
--   SIGNAL_VERY_LOW_EVIDENCE_DENSITY, SIGNAL_LOW_EVIDENCE_DENSITY: too generic for one action
--   SIGNAL_SINGLE_SOURCE_DEPENDENCE: too generic for one action
SELECT w.signal_code, w.intent, w.category, w.base_score
FROM genealogy.ref_signal_weights w
LEFT JOIN genealogy.gold_research_signal_action a ON a.signal_code = w.signal_code
WHERE a.signal_code IS NULL
  AND w.signal_code NOT IN (
    'SIGNAL_STORY_WRITTEN',
    'SIGNAL_TRANSCRIPT_RICH',
    'SIGNAL_NEWSPAPER_MENTION',
    'SIGNAL_WILL_OR_PROBATE',
    'SIGNAL_VERY_LOW_EVIDENCE_DENSITY',
    'SIGNAL_LOW_EVIDENCE_DENSITY',
    'SIGNAL_SINGLE_SOURCE_DEPENDENCE'
  )
ORDER BY w.intent, w.category;

-- 2. Action mappings pointing to non-existent codes (should return 0 rows)
SELECT sa.signal_code, sa.action_code
FROM genealogy.gold_research_signal_action sa
LEFT JOIN genealogy.gold_research_action a ON a.action_code = sa.action_code
WHERE a.action_code IS NULL;

-- 3. Confirm new DNA flag columns exist and DNA signals are present
SELECT signal_code, base_score, dimension,
       requires_dna_match_tag, requires_dna_ancestor_tag, requires_generation_lte
FROM genealogy.ref_signal_weights
WHERE signal_code IN (
  'SIGNAL_DNA_CITATION_MISSING',
  'SIGNAL_DNA_PATH_INCOMPLETE',
  'SIGNAL_NO_DNA_CORROBORATION',
  'SIGNAL_COMMON_ANCESTOR_UNVERIFIED'
)
ORDER BY signal_code;
